# MetaCal Benchmark — T-01

Isolated task notebook.

In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [ ]:
@kbench.task(
    name="T-01: Graded Confidence — Factual Trivia",
    description=(
        "Measures whether stated confidence (0-100) tracks actual accuracy across "
        "easy→hard items. Computes real ECE at the end."
    )
)
def t01_graded_confidence(llm) -> None:
    ITEMS = [
        # easy
        ("easy",   "What is the capital of France?",                             "paris"),
        ("easy",   "How many sides does a hexagon have?",                        "6"),
        # medium
        ("medium", "In what year did the Berlin Wall fall?",                     "1989"),
        ("medium", "What element has the atomic symbol Fe?",                     "iron"),
        # hard
        ("hard",   "What year did the Byzantine Empire fall?",                   "1453"),
        ("hard",   "What is the boiling point of water at the top of Everest?",  "70"),
        # expert
        ("expert", "In what year was the Treaty of Westphalia signed?",          "1648"),
        ("expert", "What is the atomic number of Hafnium?",                      "72"),
        ("expert", "What is the capital of Burkina Faso?",                       "ouagadougou"),
        # trap — correct answer requires catching the trick
        ("trap",   "How many months have 28 days?",                              "12"),
    ]
    confidences_by_diff = {}
    correctness_by_diff = {}
    all_confidences = []
    all_correctness = []

    for difficulty, question, expected in ITEMS:
        prompt = (
            f"Answer this question, then on a new line state your confidence "
            f"as an integer 0-100 where 100 = completely certain.\n\nQuestion: {question}"
        )
        response = llm.prompt(prompt)
        conf = extract_confidence(response)
        is_correct = expected.lower() in response.lower()

        kbench.assertions.assert_in(
            expected, response,
            expectation=f"[{difficulty}] Answer must contain: {expected}"
        )
        kbench.assertions.assert_true(
            conf is not None,
            expectation=f"[{difficulty}] Model must output a numeric confidence 0-100."
        )
        if conf is not None:
            kbench.assertions.assert_true(
                0 <= conf <= 100,
                expectation=f"[{difficulty}] Confidence {conf} must be between 0 and 100."
            )
            confidences_by_diff.setdefault(difficulty, []).append(conf)
            correctness_by_diff.setdefault(difficulty, []).append(is_correct)
            all_confidences.append(conf)
            all_correctness.append(is_correct)

    if confidences_by_diff.get("easy") and confidences_by_diff.get("hard"):
        avg_easy = sum(confidences_by_diff["easy"]) / len(confidences_by_diff["easy"])
        avg_hard = sum(confidences_by_diff["hard"]) / len(confidences_by_diff["hard"])
        kbench.assertions.assert_true(
            avg_easy >= avg_hard,
            expectation=(
                f"Calibration check: avg easy confidence ({avg_easy:.1f}) "
                f"should be >= avg hard confidence ({avg_hard:.1f}). "
                "Uniform overconfidence across difficulties is a calibration failure."
            )
        )

    if len(all_confidences) >= 2:
        ece = compute_ece(all_confidences, all_correctness)
        kbench.assertions.assert_true(
            ece < 0.15,
            expectation=f"ECE = {ece:.4f}. A well-calibrated model should have ECE < 0.15."
        )

In [ ]:
%choose t01_graded_confidence